# Beginner tour 6: Smarter strategies (the `adaptive_acq` project)

Notebook 4 found that exploring with a **fixed** amount (UCB with β = 2) didn't beat greedy.
The project's follow-up, the `adaptive_acq/` folder, asked a natural next question:

> What if the amount of exploration **changes every round**, chosen by a small decision-maker
> (a **controller**) that watches how the campaign is going?

That's a classic idea from reinforcement learning, called a **bandit**. In this notebook you'll
run the follow-up project's real code: its `Campaign` class and its controllers. You'll also
meet two Python ideas that make the design work: **functions passed as values**, and
**classes that share an interface**.

Spoiler: the project's honest answer was *no, nothing beats greedy*. Seeing **why**, and how
the researchers then found improvements elsewhere, is the most useful lesson in research
practice in this tour.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tour import *

scores, embeddings = load_mini()
y = scores["score"].to_numpy()                 # NOT flipped here: Campaign's score_fn returns docking scores (lower = better)
N = len(y)
true_top = np.zeros(N, dtype=bool)
true_top[np.argsort(y)[:100]] = True           # True for the best 100, used ONLY for grading
print(N, "molecules,", true_top.sum(), "marked as the best")

🐍 **Python corner: boolean masks.** `true_top` is an array of `True`/`False`, one per
molecule. `np.zeros(N, dtype=bool)` starts all `False`, and `true_top[positions] = True`
switches some on. Masks like this are everywhere in the project: "which molecules are
labeled", "which are candidates".

---
## 1. A fast stand-in model

A controller has to be tested over **many** campaigns. So `adaptive_acq` has a fast
simulator (`sim_env.py`) with two shortcuts, which we copy:

* **Shrink the embeddings** from 768 numbers to 64 with **PCA**, a standard method that keeps
  the most informative directions. (No need to follow the maths: think of it as compressing a
  photo.)
* Use a **cheap surrogate**, `BootstrapRidge`: 5 straight-line models, each trained on a
  slightly different random resample of the docked molecules. Their **disagreement** is
  `sigma`.

In [ ]:
X = embeddings["unimol2"].astype(np.float64)
X = (X - X.mean(0)) / (X.std(0) + 1e-6)
U, S, _ = np.linalg.svd(X - X.mean(0), full_matrices=False)    # PCA via the SVD
X64 = U[:, :64] * S[:64]
print("from", X.shape, "to", X64.shape, f"- keeping {(S[:64]**2).sum() / (S**2).sum():.1%} of the information")

---
## 2. Functions as values: how `Campaign` is plugged together

`Campaign` (in `adaptive_acq/campaign.py`) runs one active-learning campaign. It was written
so that **the same code** runs both the cheap simulator and the real 2-million-molecule
experiment. How? It doesn't know or care what the model is or how docking works. You **hand
it two functions**:

* `fit_predict(labeled_idx, y_hb, cand_idx)`: "train on these, guess those" → `(mu, sigma)`
* `score_fn(idx)`: "dock these" → their docking scores

🐍 **Python corner: functions are values.** In Python you can store a function in a variable
and pass it to another function, like any number or list. A **`lambda`** is a short,
nameless function written in one line: `lambda a, b: a + b` is a function that adds two
things.

In [ ]:
def apply_twice(f, x):        # takes a FUNCTION f as its first input
    return f(f(x))

print(apply_twice(lambda v: v * 10, 3))           # (3*10)*10
dock = lambda idx: y[idx]                          # "docking" = looking up the oracle
print("docking molecules 0, 1, 2:", dock(np.array([0, 1, 2])))

🐍 **Python corner: closures.** A `lambda` can use variables from where it was written. Below,
`fit_predict` "remembers" `surrogate` and `X64`, even though `Campaign` only ever passes it
three arguments. That's a **closure**, and it's how the real `AdaptiveMVEExplorer` connects
its LT-All model to `Campaign` too.

In [ ]:
from adaptive_acq.campaign import Campaign
from adaptive_acq.controllers import Controller, FixedBeta, Schedule, BatchSplitThompson
from adaptive_acq.sim_env import BootstrapRidge

def make_campaign(seed, prefilter_mult=0.0):
    surrogate = BootstrapRidge(M=5, alpha=10.0, seed=seed)
    start = np.random.default_rng(seed).choice(N, 105, replace=False)       # 0.5% of the pool, at random
    return Campaign(
        n_pool=N,
        fit_predict=lambda labeled_idx, y_hb, cand_idx: surrogate(X64, labeled_idx, y_hb, cand_idx),
        score_fn=lambda idx: y[idx],
        init_idx=start, batch_size=35, n_rounds=9,     # a TIGHT budget: 105 + 9 x 35 = 420 molecules (2%)
        k_obs=100, true_top_mask=true_top, prefilter_mult=prefilter_mult,
    )

history = make_campaign(seed=0).run(FixedBeta(0.0))   # beta = 0 means greedy
pd.DataFrame(history)[["round", "n_labeled", "best_score", "topk_recall", "reward", "beta_mean"]]

Columns you know (`n_labeled`, `topk_recall`), plus two new ones:

* `reward`: what fraction of this round's batch were **hits**. A hit is a molecule in the
  estimated top 1% of the library. This is what a controller learns from. Note that it uses
  only information a real screen would have, never the true best-100 list.
* `beta_mean`: how much exploration was used (0 = pure greedy).

We used a **tight** budget (2% of the pool) on purpose. Notebook 2 showed that with a
generous budget everything hits the ceiling and no strategy can look better than another.

---
## 3. What a controller sees

Each round, `Campaign` summarizes the situation in **10 numbers** called the **context** and
shows them to the controller. The controller never sees molecules, only this summary:

In [ ]:
from adaptive_acq.state import FEATURE_NAMES
pd.DataFrame([r["context"] for r in history], columns=FEATURE_NAMES, index=[f"round {r['round']}" for r in history]).round(2)

A few of them in plain words:
* `progress`: how far through the campaign we are (0 → 1)
* `last_reward`: how good the last batch was
* `last_spearman`: did the model's guesses on the last batch get the order right?
* `last_sigma_err_corr`: was the model unsure in the right places? (If not, exploring on its uncertainty is pointless.)
* `ucb_greedy_jaccard`: how much would exploring actually change the batch?

---
## 4. Controllers: classes that share an interface

Every controller is a **class** with the same two methods:
* `plan(...)`: decide this round's exploration amount β (it may split the batch between several βs)
* `update(...)`: learn from how many hits the batch got

Here's the base class and the simplest controller, straight from `adaptive_acq/controllers.py`:

In [ ]:
import inspect
print(inspect.getsource(Controller.plan))
print(inspect.getsource(FixedBeta))

🐍 **Python corner: classes, briefly.** `class FixedBeta(Controller):` means "FixedBeta is a
kind of Controller". It **inherits** everything from `Controller` and **replaces** `plan`.
The base class's `plan` just raises `NotImplementedError`, which is a way of saying "every
controller must write its own". `__init__` runs when you create one: `FixedBeta(0.3)` stores
`self.beta = 0.3`. `classes_and_call_flow.ipynb` covers classes in depth.

A plan is a list of `(arm, beta, count)` entries: "use this β for this many molecules". Because
every controller has the same `plan`/`update` methods, `Campaign` can run **any** of them,
**including one you write yourself**:

In [ ]:
class ExploreEarlyOnly(Controller):
    # "explore a lot in the first 3 rounds, then be greedy"
    name = "mine"
    arms = (0.0, 0.5)

    def plan(self, ctx, rnd, n_rounds, batch_size):
        beta = 0.5 if rnd < 3 else 0.0
        return [(0, beta, batch_size)]          # one arm, all molecules

h = make_campaign(seed=0).run(ExploreEarlyOnly())
print("beta used each round:", [r["beta_mean"] for r in h])
print("final recall:", h[-1]["topk_recall"])

---
## 5. The contest

Five strategies, 5 seeds each, all with the same tight budget:

| Controller | Behaviour |
|---|---|
| `FixedBeta(0.0)` | greedy, never explore |
| `FixedBeta(0.3)` | always explore a little |
| `FixedBeta(1.0)` | always explore a lot |
| `Schedule("explore_then_exploit")` | explore in the first half, then greedy |
| `BatchSplitThompson()` | a **bandit**: splits each batch across 6 β values, and gives more of the next batch to whichever β found more hits |
| `ExploreEarlyOnly()` | yours, from above |

(β here is "normalized": 1 means an average uncertainty bonus is worth one standard
deviation of the guesses, so the numbers mean the same for any model.)

In [ ]:
contenders = {
    "greedy (beta 0)":      lambda s: FixedBeta(0.0),
    "beta 0.3":             lambda s: FixedBeta(0.3),
    "beta 1.0":             lambda s: FixedBeta(1.0),
    "explore, then exploit": lambda s: Schedule("explore_then_exploit", beta_hi=0.5, beta_lo=0.0),
    "Thompson bandit":      lambda s: BatchSplitThompson(seed=s),
    "yours":                lambda s: ExploreEarlyOnly(),
}
results = {name: [make_campaign(s).run(make(s))[-1]["topk_recall"] for s in range(5)]
           for name, make in contenders.items()}

table = pd.DataFrame({n: {"mean recall": np.mean(r), "spread": np.std(r)} for n, r in results.items()}).T
display(table.sort_values("mean recall", ascending=False).round(3))

🐍 **Python corner: a dictionary comprehension.** `{name: ... for name, make in
contenders.items()}` builds a dictionary in one line, the same way a list comprehension
builds a list.

**Greedy is at the top, or tied for it.** The cleverer controllers, including the learning
bandit, end up about level with it or below, and exploring a lot (β = 1) is clearly worse.
This is exactly what the project found at full scale:

> 145 pre-registered runs established that no choice of β, no schedule and no learned
> controller beats greedy. (`al-molecular/docs/adaptive_acq_results.md`)

### Watching the bandit learn

The bandit's history records how it split each batch across the 6 β values. It starts with
an even split, quickly gives the biggest β (0.5) only the minimum 2 molecules, and moves the
rest around among the small βs (0 to 0.2):

In [ ]:
h = make_campaign(seed=0).run(BatchSplitThompson(seed=0))
shares = pd.DataFrame([{v["beta"]: v["n"] for v in r["plan"].values()} for r in h], index=range(1, 10))
shares.plot(kind="bar", stacked=True, figsize=(7, 3.5), colormap="viridis", width=0.85)
plt.xlabel("round"); plt.ylabel("molecules in the batch"); plt.legend(title="beta", bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()

Notice how much the split jumps from round to round. Its feedback is noisy: 35 molecules
per round, split six ways, is very little evidence per β. That's part of why learning *how much* to explore couldn't win. There's
little to gain, and little evidence to learn from.

---
## 6. Where the real improvements came from

Good research reports negative results and then moves on. After acquisition turned out to be
a dead end, the project found three techniques that did help on the full pool, all **elsewhere
in the loop**:

**More, smaller rounds.** You saw this in notebook 2.

**The prefilter: score everything once, then only the promising part.** Predicting all 2
million (or 1 billion) molecules every round is the most expensive step. The prefilter
scores everything in round 1, keeps a **shortlist** (a multiple of the budget, e.g. 10×),
and scores only the shortlist after that. `Campaign` has it built in (`prefilter_mult`):

In [ ]:
for mult in [0, 10, 3]:
    histories = [make_campaign(s, prefilter_mult=mult).run(FixedBeta(0.0)) for s in range(5)]
    recall = np.mean([h[-1]["topk_recall"] for h in histories])
    scored = np.mean([sum(r["n_scored"] for r in h) for h in histories])
    label = "no prefilter" if mult == 0 else f"shortlist {mult}x the budget"
    print(f"{label:28s} recall {recall:.3f}   molecules scored per campaign: {scored:9,.0f}")

Same recall, a fraction of the work. At billion-molecule scale that's the difference between
feasible and not. The project's best configuration combines 20 rounds with this prefilter:
+0.014 recall at 71% of the compute.

**Top-weighted training (`top_weight`).** The surrogate is normally trained to be accurate
on *all* molecules, but we only ever pick from the top. `LTAllSurrogate(top_weight=1.0)`
weights each training molecule by `exp(top_weight × standardized score)`, so getting the good
ones right counts more:

In [ ]:
y_std = np.array([-2, -1, 0, 1, 2, 3])           # standardized (flipped) scores: 3 = excellent
for tw in [0.0, 1.0, 3.0]:
    w = np.exp(tw * y_std); w = w / w.mean()
    print(f"top_weight {tw}: weights", w.round(2))

With `top_weight = 1`, an excellent molecule (standardized score 3) counts about 20× more
than an average one (0), and over 100× more than a poor one (−2). At `top_weight = 3`
nearly all the weight sits on the very best few molecules, and the model is effectively
trained on a handful of examples. The project found that this collapses.

This was the project's **one reliable recall technique**. It was chosen on one set of seeds,
then held up on new seeds and on a second protein (D4). An honest caveat from trying it on
this mini pool: with the tight budget above, it did *not* help (about 0.65 without vs 0.62
with, over 5 seeds; each run takes about 40 seconds, so it isn't run here). On the full
ENHITS pool its gain was small (+0.004 to +0.015), and on the harder D4 target it was large
(+0.055). **A technique can help in one setting and not show up in another.** That's why
the project tested on two targets and fresh seeds before calling it reliable.

---
## Summary

* `adaptive_acq` asked whether a **controller** that changes exploration each round beats
  greedy. On real data here, and in 145 runs at full scale: **no**.
* `Campaign` takes its model and its docking step as **functions**, so the same loop runs the
  simulator and the real experiment.
* Controllers are **classes with a shared interface** (`plan`, `update`), so you could write
  your own and plug it in.
* The real gains came from **elsewhere**: more rounds, a prefilter that cuts cost, and
  top-weighted training.

🐍 **Python you met:** boolean masks, functions as values, `lambda`, closures, a class that
inherits and overrides, `NotImplementedError`, dictionary comprehensions, stacked bar charts.

## Try it yourself
Answers, run on the same real data: [solutions/6_solutions.ipynb](solutions/6_solutions.ipynb). Try first!

1. Make your controller explore only in the **last** 3 rounds. Better or worse than "early"?
2. Change `make_campaign` to a generous budget (`batch_size=210, n_rounds=5`, start with 210). Do the controllers still differ?
3. Try `prefilter_mult=1`. When does the shortlist become too small?

**Next:** `7_scaling_up.ipynb`, from 21,000 molecules to 100 million.